# 28 · Relate a table against a pandas merge

**Question.** Snow's Table XII says which water company supplied each of 32 sub-districts. Does
bringing that company onto the 64 rows of his Table VIII land the same category on the same row
as an independent left join — and does it do the right thing where a join does the wrong thing?

**There is no arithmetic here, and that is the point.** A relate brings a *category* across; no
number is computed. So this is not a check on a sum. It is a check on the **matching**, and on
one case where the two implementations are meant to disagree.

`pandas.DataFrame.merge` is the operation a practitioner would otherwise reach for, written by
other authors. Where it agrees, the agreement is worth having. Where it does not, the difference
is the finding.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

Three cases, all recomputed from the fixture's two input tables and the parameters it declares:

1. **The relate the example runs** — Table XII's company onto all 64 rows of Table VIII.
   Compared as a set of records, so row order is not mistaken for disagreement, and the row count
   must be **unchanged**: a relate that multiplied rows would be a merge.
2. **A key the right table does not carry** — one sub-district withheld. Both drop its rows;
   Fieldwork also names the key and counts the rows, and the names are compared.
3. **An ambiguous key** — one sub-district given a second, different company. This is the case
   worth an outside eye, and it is *constructed*: no sub-district in Table XII carries two
   companies, which is exactly why the case cannot be drawn from the data.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_28_relate_table import run

report = run(Path.cwd() / "fixtures" / "relate-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}")
    print(f"  rows   Fieldwork {c['fieldwork']['rowCount']:3d}   pandas {c['pandas']['rowCount']:3d}")
    print(f"  dropped          {c['fieldwork']['dropped']:3d}          {c['pandas']['dropped']:3d}")
    print(f"  gaps   {c['fieldwork']['gaps']}")
    for name, ok in c["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The case worth the check: an ambiguous key

A right table that gives one key two different values is **refused** here, with a message naming
the key and both values. A merge does not refuse. It **multiplies** the rows — each left row for
that key matches twice — so the stratification quietly carries that sub-district's population
under both companies, and no total in the result reveals it.

The cell below prints the refusal and, beside it, how many rows the merge returns for the same
input. That number is what the refusal buys.


In [ ]:
a = report["ambiguousKey"]
print("key              :", a["key"])
print("two values       :", " / ".join(a["values"]))
print("left rows for it :", a["leftRowsForThatKey"])
print()
print("Fieldwork refuses:")
print(" ", a["fieldworkRefused"])
print()
print("pandas merge returns:", a["pandasRowCount"], "rows")
print(" ", a["whatTheRefusalBuys"])
print()
for name, ok in a["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")


## What this cannot establish

**That exact matching is the right matching.** Snow's two tables spell every sub-district
identically, so this data never exercises a name that differs by punctuation or case —
*"St. George, Camberwell"* against *"St George Camberwell"* is the case a practitioner will
actually meet, and nothing here reaches it. Fieldwork folds surrounding whitespace and nothing
else, by decision rather than by this result.

**That relating is right for bringing measures across.** That adds rows rather than a column,
has a different failure mode, and is deliberately not built.

**That the category is the right stratifier.** The both-companies group is 300,149 of the 486,936
people in the table. What to do with it is the practitioner's judgement, not an outcome of the
join — and it is Snow's own confounding, which is why he went to the house level.


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
